# preprocessing

In [ ]:
import os
import random

class LOLDatasetLoader:
    def __init__(self, low_dir, high_dir, target_size=(256, 256), augment=False):

        self.low_dir = low_dir
        self.high_dir = high_dir
        self.target_size = target_size
        self.augment = augment

        files_low = sorted(os.listdir(low_dir))
        files_high = sorted(os.listdir(high_dir))

        self.low_images_path = files_low[:100]
        self.high_images_path = files_high[:100]

        print(f"successful to load {len(self.low_images_path)} images")


    def load_and_preprocess(self):

        low_images = []
        high_images = []

        for low_path, high_path in zip(self.low_images_path, self.high_images_path):

            low = os.path.join(self.low_dir, low_path)
            high = os.path.join(self.high_dir, high_path)

            low_img = cv2.imread(low, cv2.COLOR_BGR2RGB)
            high_img = cv2.imread(high, cv2.COLOR_BGR2RGB)

            low_img = cv2.cvtColor(low_img, cv2.COLOR_BGR2RGB)
            high_img = cv2.cvtColor(high_img, cv2.COLOR_BGR2RGB)


            low_img = cv2.resize(low_img, self.target_size)
            high_img = cv2.resize(high_img, self.target_size)

            low_img = low_img.astype(np.float32) / 255.0
            high_img = high_img.astype(np.float32) / 255.0

            low_images.append(low_img)
            high_images.append(high_img)


            if self.augment and random.random() >= 0.5:
                low_img = cv2.flip(low_img, 1) 
                high_img = cv2.flip(high_img, 1)

        return np.array(low_images), np.array(high_images)



LOW_DIR = "LOL-v2/Real_captured/Train/Low"
HIGH_DIR = "LOL-v2/Real_captured/Train/Normal"


loader = LOLDatasetLoader(LOW_DIR, HIGH_DIR, (256, 256), False)

low_imgs, high_imgs = loader.load_and_preprocess()

print(f"Shape of Low Images Array: {low_imgs.shape}")   
print(f"Shape of High Images Array: {high_imgs.shape}") 

# PAHSE 1

## create Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay)

def extract_feature(image):

    img = (image * 255).astype(np.uint8)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mean_intensity = np.mean(gray)
    std_deviation = np.std(gray)

    hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).ravel()
    hist_prob = hist / hist.sum()
    hist_prob = hist_prob[hist_prob > 0]
    entropy = -np.sum(hist_prob * np.log2(hist_prob))

    skewness = skew(gray.ravel())

    return [mean_intensity, std_deviation, entropy, skewness]


X = []
y = []

for img in low_imgs:
    X.append(extract_feature(img))
    y.append(0)

for img in high_imgs:
    X.append(extract_feature(img))
    y.append(1)

X = pd.DataFrame(X, columns=["mean_intensity", "std_deviation", "entropy", "skewness"])
y = np.array(y)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

## train a model

In [ ]:
classifier = LogisticRegression(random_state=42)
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("-" * 30)
print("نتایج ارزیابی مدل Logistic Regression:")
print(f"Accuracy  (دقت کل):        {acc:.4f}")
print(f"Precision (دقت مثبت):      {prec:.4f}")
print(f"Recall    (فراخوانی):      {rec:.4f}")
print(f"F1-Score  (معیار ترکیبی):   {f1:.4f}")
print("-" * 30)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Low-Light (0)", "Normal (1)"])

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title("Confusion Matrix for Image Lighting Classification")
plt.show()


# PHASE 2

## A)

In [ ]:
import matplotlib.pyplot as plt

def apply_he(image_rgb_norm):
    img_uint8 = (image_rgb_norm * 255).astype(np.uint8)
    ycrcb = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2YCrCb)
    channels = cv2.split(ycrcb)
    cv2.equalizeHist(channels[0], channels[0])
    cv2.merge(channels, ycrcb)
    result = cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2RGB)
    return result / 255.0

def apply_clahe(image_rgb_norm, clip_limit=2.0, tile_grid_size=(8, 8)):
    img_uint8 = (image_rgb_norm * 255).astype(np.uint8)
    ycrcb = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2YCrCb)
    channels = list(cv2.split(ycrcb))
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    channels[0] = clahe.apply(channels[0]) 
    ycrcb = cv2.merge(channels)
    result = cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2RGB)
    return result / 255.0

def apply_gamma(image_rgb_norm, gamma=0.5):
    result = np.power(image_rgb_norm, gamma)
    return np.clip(result, 0, 1)

def apply_ssr(image_rgb_norm, sigma=80):
    epsilon = 1e-6
    img = image_rgb_norm + epsilon
    log_img = np.log10(img)
    img_blur = cv2.GaussianBlur(img, (0, 0), sigma)
    log_blur = np.log10(img_blur + epsilon)
    retinex = log_img - log_blur
    for i in range(3):
        retinex[:, :, i] = cv2.normalize(retinex[:, :, i], None, 0, 1, cv2.NORM_MINMAX)

    return retinex


sample_idx = 0
original = low_imgs[sample_idx]
ref_normal = high_imgs[sample_idx]

res_he = apply_he(original)
res_clahe = apply_clahe(original)
res_gamma_04 = apply_gamma(original, gamma=0.4) 
res_ssr = apply_ssr(original, sigma=30)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

titles = ["Input Low-Light", "Normal Reference", "Histogram Eq (HE)", "CLAHE", "Gamma (0.4)", "Retinex (SSR)"]
images = [original, ref_normal, res_he, res_clahe, res_gamma_04, res_ssr]

for i, ax in enumerate(axes):
    ax.imshow(images[i])
    ax.set_title(titles[i])
    ax.axis('off')

plt.tight_layout()
plt.show()


## B

In [ ]:
def create_loaders_from_variables(low_tr, high_tr, low_te, high_te, batch_size=10):
    def process_data(data, is_low_light=False):  
        if not torch.is_tensor(data):
            data = torch.tensor(data, dtype=torch.float32)
        else:
            data = data.float()

        if data.ndim == 4 and data.shape[-1] == 3:
            data = data.permute(0, 3, 1, 2)

        if data.max() > 1.0:
            data = data / 255.0
        
        return data

    print("در حال آماده‌سازی داده‌های تصویری برای Autoencoder...")
    
    low_train_tensor = process_data(low_tr, is_low_light=True)
    high_train_tensor = process_data(high_tr, is_low_light=False)
    low_test_tensor = process_data(low_te, is_low_light=True)
    high_test_tensor = process_data(high_te, is_low_light=False)

    print(f"ابعاد ورودی شبکه (Train): {low_train_tensor.shape}")

    train_dataset = TensorDataset(low_train_tensor, high_train_tensor)
    test_dataset = TensorDataset(low_test_tensor, high_test_tensor)

    train_loader_obj = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader_obj = DataLoader(test_dataset, batch_size=1, shuffle=False)

    return train_loader_obj, test_loader_obj


low_train, low_test, high_train, high_test = train_test_split(
    low_imgs, high_imgs, test_size=0.2, random_state=42
)

train_loader, test_loader = create_loaders_from_variables(
    low_train, high_train, low_test, high_test
)


## Create Autoencoder

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchmetrics.image import StructuralSimilarityIndexMeasure
import copy


class ImprovedAutoencoder(nn.Module):
    def __init__(self):
        super(ImprovedAutoencoder, self).__init__()
        
        # ===== Encoder =====
        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.LeakyReLU(0.2), nn.BatchNorm2d(64)
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.LeakyReLU(0.2), nn.BatchNorm2d(128),
        )
        self.enc3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.LeakyReLU(0.2), nn.BatchNorm2d(256),
        )
        self.enc4 = nn.Sequential(
            nn.Conv2d(256, 512, 3, padding=1), nn.LeakyReLU(0.2),nn.BatchNorm2d(512)
        )
        
        self.pool = nn.MaxPool2d(2, 2)

        self.bottleneck = nn.Sequential(
            nn.Conv2d(512, 768, 3, padding=1), nn.BatchNorm2d(768) ,nn.LeakyReLU(0.2),
            nn.Conv2d(768, 1024, 3, padding=1), nn.BatchNorm2d(1024) ,nn.LeakyReLU(0.2),
            nn.Conv2d(1024, 1024, 3, padding=1), nn.BatchNorm2d(1024) ,nn.LeakyReLU(0.2),
            nn.Conv2d(1024, 512, 3, padding=1), nn.BatchNorm2d(512) ,nn.LeakyReLU(0.2),
        )

    
        self.up4 = nn.ConvTranspose2d(512, 512, 2, stride=2)
        self.dec4 = nn.Sequential(
            nn.Conv2d(1024, 512, 3, padding=1),  
            nn.LeakyReLU(0.2),
            nn.BatchNorm2d(512)
        )
        
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = nn.Sequential(
            nn.Conv2d(512, 256, 3, padding=1),  
            nn.LeakyReLU(0.2),
            nn.BatchNorm2d(256),
        )
        
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1),  
            nn.LeakyReLU(0.2),
            nn.BatchNorm2d(128),
        )
        
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1),   
            nn.LeakyReLU(0.2),
            nn.BatchNorm2d(64),
        )
        
        self.final = nn.Sequential(
            nn.Conv2d(64, 3, 3, padding=1), 
            nn.Sigmoid(),
        )
        
    def forward(self, x):
        e1 = self.enc1(x)          
        e2 = self.enc2(self.pool(e1)) 
        e3 = self.enc3(self.pool(e2)) 
        e4 = self.enc4(self.pool(e3))  
        
        b = self.bottleneck(self.pool(e4))   
        
        d4 = self.up4(b)            
        d4 = torch.cat([d4, e4], dim=1)  
        d4 = self.dec4(d4)          
        
        d3 = self.up3(d4)          
        d3 = torch.cat([d3, e3], dim=1)  
        d3 = self.dec3(d3)          
        
        d2 = self.up2(d3)           
        d2 = torch.cat([d2, e2], dim=1)  
        d2 = self.dec2(d2)          
        
        d1 = self.up1(d2)           
        d1 = torch.cat([d1, e1], dim=1)  
        d1 = self.dec1(d1)          
        
        return self.final(d1)       
 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImprovedAutoencoder().to(device)

ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0002, weight_decay=1e-5)


scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

criterion = nn.MSELoss()

best_loss = float('inf')
patience = 10
counter = 0
train_loss = []

for epoch in range(30):
    model.train()
    epoch_loss = 0.0
    
    for low, high in train_loader:
        low, high = low.to(device), high.to(device)
        
        optimizer.zero_grad()
        out = model(low)
        
        out = torch.clamp(out, 0, 1)
        
        loss = criterion(out, high)
        # loss = ssim_loss(out, high)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        epoch_loss += loss.item() * low.size(0)
    
    epoch_loss /= len(train_loader.dataset)
    train_loss.append(epoch_loss)
    scheduler.step(epoch_loss)
    
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        best_model = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    print(f"Epoch {epoch+1}/30, Loss: {epoch_loss:.4f}, LR: {optimizer.param_groups[0]['lr']:.6f}")

print("best_loss: ", best_loss)
model.load_state_dict(best_model)

## Evaluate Autoencoder

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_loss) + 1), train_loss, marker='o', color='b', label='Training Loss (MSE)')
plt.title('Training Loss Curve (Autoencoder)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

print("در حال استخراج یک تصویر تست برای مشاهده عملکرد مدل...")



model.eval()


for i in range(len(test_loader.dataset)):

    low_batch, high_batch = test_loader.dataset[i]
    low_batch = low_batch.unsqueeze(0).to(device)
    high_batch = high_batch.unsqueeze(0).to(device)

    print(f"Shape of low_batch: {low_batch.shape}")
    print(f"Shape of high_batch: {high_batch.shape}")


    with torch.no_grad():
        raw_output = model(low_batch)
        raw_output = torch.clamp(raw_output, 0, 1)

    def tensor_to_image(tensor):
        tensor = tensor.cpu()
        img = tensor.squeeze(0)
        img = img.numpy()
        img = np.transpose(img, (1, 2, 0))
        img = np.clip(img, 0, 1)
        return img


    plt.figure(figsize=(16, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(tensor_to_image(low_batch[0]))
    plt.title("Input (Low-Light)")
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(tensor_to_image(raw_output[0]))
    plt.title("Raw Model Output")
    plt.axis('off')


    plt.subplot(1, 3, 3)
    plt.imshow(tensor_to_image(high_batch[0]))
    plt.title("Ground Truth (Normal-Light)")
    plt.axis('off')

    plt.tight_layout()
    plt.show()


    from skimage.metrics import peak_signal_noise_ratio as psnr
    from skimage.metrics import structural_similarity as ssim

    low_img = tensor_to_image(raw_output[0])
    high_img = tensor_to_image(high_batch[0])


    psnr_value = psnr(high_img, low_img, data_range=1.0)
    ssim_value = ssim(high_img, low_img, data_range=1.0, channel_axis=2)


    print(f"\n📊 Metrics for this sample:")
    print(f"PSNR: {psnr_value:.2f} dB")
    print(f"SSIM: {ssim_value:.4f}")



## Evaluatation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
from skimage.metrics import peak_signal_noise_ratio as calculate_psnr
from skimage.metrics import structural_similarity as calculate_ssim

results = {
    "Original Low": {"psnr": [], "ssim": []},
    "HE": {"psnr": [], "ssim": []},
    "CLAHE": {"psnr": [], "ssim": []},
    "Gamma": {"psnr": [], "ssim": []},
    "Autoencoder": {"psnr": [], "ssim": []},
}

print("در حال ارزیابی روش‌ها روی داده‌های تست... لطفا صبر کنید.")


model.eval()
with torch.no_grad():
    for i in range(len(low_test)):
        low_img = low_test[i]
        high_img = high_test[i]

        results["Original Low"]["psnr"].append(calculate_psnr(high_img, low_img, data_range=1.0))
        results["Original Low"]["ssim"].append(calculate_ssim(high_img, low_img, data_range=1.0, channel_axis=2))


        he_img = apply_he(low_img)
        results["HE"]["psnr"].append(calculate_psnr(high_img, he_img, data_range=1.0))
        results["HE"]["ssim"].append(calculate_ssim(high_img, he_img, data_range=1.0, channel_axis=2))

        clahe_img = apply_clahe(low_img)
        results["CLAHE"]["psnr"].append(calculate_psnr(high_img, clahe_img, data_range=1.0))
        results["CLAHE"]["ssim"].append(calculate_ssim(high_img, clahe_img, data_range=1.0, channel_axis=2))

        gamma_img = apply_gamma(low_img)
        results["Gamma"]["psnr"].append(calculate_psnr(high_img, gamma_img, data_range=1.0))
        results["Gamma"]["ssim"].append(calculate_ssim(high_img, gamma_img, data_range=1.0, channel_axis=2))

        low_tensor = torch.tensor(np.transpose(low_img, (2, 0, 1)), dtype=torch.float32).unsqueeze(0).to(device)
        dl_output = model(low_tensor)
        dl_img = dl_output.squeeze(0).cpu().numpy()
        dl_img = np.transpose(dl_img, (1, 2, 0)) 

        results["Autoencoder"]["psnr"].append(calculate_psnr(high_img, dl_img, data_range=1.0))
        results["Autoencoder"]["ssim"].append(calculate_ssim(high_img, dl_img, data_range=1.0, channel_axis=2))
        

methods = list(results.keys())
avg_psnr = [np.mean(results[m]["psnr"]) for m in methods]
avg_ssim = [np.mean(results[m]["ssim"]) for m in methods]

print("\n" + "="*45)
print(f"{'Method':<15} | {'Avg PSNR (dB)':<12} | {'Avg SSIM':<10}")
print("="*45)
for i, m in enumerate(methods):
    print(f"{m:<15} | {avg_psnr[i]:<12.2f} | {avg_ssim[i]:<10.4f}")
print("="*45)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(methods, avg_psnr, color=['gray', 'blue', 'cyan', 'green', 'red'])
ax1.set_title('Average PSNR Comparison (Higher is Better)')
ax1.set_ylabel('PSNR (dB)')
ax1.grid(axis='y', linestyle='--', alpha=0.7)
for i, v in enumerate(avg_psnr):
    ax1.text(i, v + 0.5, f"{v:.2f}", ha='center')

ax2.bar(methods, avg_ssim, color=['gray', 'blue', 'cyan', 'green', 'red'])
ax2.set_title('Average SSIM Comparison (Closer to 1 is Better)')

ax2.set_ylabel('SSIM')
ax2.grid(axis='y', linestyle='--', alpha=0.7)
for i, v in enumerate(avg_ssim):
    ax2.text(i, v + 0.02, f"{v:.3f}", ha='center')

plt.tight_layout()
plt.show()


In [ ]:
LOW_DIR = "LOL-v2/Real_captured/Test/Low"
HIGH_DIR = "LOL-v2/Real_captured/Test/Normal"


loader = LOLDatasetLoader(LOW_DIR, HIGH_DIR, (256, 256), False)

low_imgs, high_imgs = loader.load_and_preprocess()



results = {
    "Original Low": {"psnr": [], "ssim": []},
    "HE": {"psnr": [], "ssim": []},
    "CLAHE": {"psnr": [], "ssim": []},
    "Gamma": {"psnr": [], "ssim": []},
    "Autoencoder": {"psnr": [], "ssim": []},
}

print("در حال ارزیابی روش‌ها روی داده‌های تست... لطفا صبر کنید.")


model.eval()
with torch.no_grad():
    for i in range(len(low_imgs)):
        low_img = low_imgs[i]
        high_img = high_imgs[i]

        results["Original Low"]["psnr"].append(calculate_psnr(high_img, low_img, data_range=1.0))
        results["Original Low"]["ssim"].append(calculate_ssim(high_img, low_img, data_range=1.0, channel_axis=2))


        he_img = apply_he(low_img)
        results["HE"]["psnr"].append(calculate_psnr(high_img, he_img, data_range=1.0))
        results["HE"]["ssim"].append(calculate_ssim(high_img, he_img, data_range=1.0, channel_axis=2))

        clahe_img = apply_clahe(low_img)
        results["CLAHE"]["psnr"].append(calculate_psnr(high_img, clahe_img, data_range=1.0))
        results["CLAHE"]["ssim"].append(calculate_ssim(high_img, clahe_img, data_range=1.0, channel_axis=2))

        gamma_img = apply_gamma(low_img)
        results["Gamma"]["psnr"].append(calculate_psnr(high_img, gamma_img, data_range=1.0))
        results["Gamma"]["ssim"].append(calculate_ssim(high_img, gamma_img, data_range=1.0, channel_axis=2))

        low_tensor = torch.tensor(np.transpose(low_img, (2, 0, 1)), dtype=torch.float32).unsqueeze(0).to(device)
        dl_output = model(low_tensor)
        dl_img = dl_output.squeeze(0).cpu().numpy()
        dl_img = np.transpose(dl_img, (1, 2, 0)) 

        results["Autoencoder"]["psnr"].append(calculate_psnr(high_img, dl_img, data_range=1.0))
        results["Autoencoder"]["ssim"].append(calculate_ssim(high_img, dl_img, data_range=1.0, channel_axis=2))
        

methods = list(results.keys())
avg_psnr = [np.mean(results[m]["psnr"]) for m in methods]
avg_ssim = [np.mean(results[m]["ssim"]) for m in methods]

print("\n" + "="*45)
print(f"{'Method':<15} | {'Avg PSNR (dB)':<12} | {'Avg SSIM':<10}")
print("="*45)
for i, m in enumerate(methods):
    print(f"{m:<15} | {avg_psnr[i]:<12.2f} | {avg_ssim[i]:<10.4f}")
print("="*45)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(methods, avg_psnr, color=['gray', 'blue', 'cyan', 'green', 'red'])
ax1.set_title('Average PSNR Comparison (Higher is Better)')
ax1.set_ylabel('PSNR (dB)')
ax1.grid(axis='y', linestyle='--', alpha=0.7)
for i, v in enumerate(avg_psnr):
    ax1.text(i, v + 0.5, f"{v:.2f}", ha='center')

ax2.bar(methods, avg_ssim, color=['gray', 'blue', 'cyan', 'green', 'red'])
ax2.set_title('Average SSIM Comparison (Closer to 1 is Better)')

ax2.set_ylabel('SSIM')
ax2.grid(axis='y', linestyle='--', alpha=0.7)
for i, v in enumerate(avg_ssim):
    ax2.text(i, v + 0.02, f"{v:.3f}", ha='center')

plt.tight_layout()
plt.show()
